# 🌙 01. Chandrayaan-2 Lunar Dataset Quality & Statistical Analysis

**Mission Context**: ISRO Chandrayaan-2 Orbiter Payloads (OHRC, TMC-2, IIR) & Reference Datasets (LROC QuickMap).  
**Objective**: Scan raw image directories, verify file integrity, detect duplicates via MD5 hashing, identify corrupted frames, calculate radiometric brightness and RMS contrast distributions, and export comprehensive audit reports.

---
### Mathematical Formulations:
1. **Mean Radiometric Brightness**:
   $$\mu = \frac{1}{N} \sum_{i=1}^N I(x_i, y_i)$$
2. **RMS Contrast**:
   $$C_{RMS} = \sqrt{\frac{1}{N} \sum_{i=1}^N (I(x_i, y_i) - \mu)^2}$$
3. **Michelson Contrast**:
   $$C_M = \frac{I_{max} - I_{min}}{I_{max} + I_{min} + \epsilon}$$


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import json

# Add project root to path
sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.data_loader import LunarDatasetScanner, LunarImageLoader
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()
print("Configuration loaded. Target root:", config.project_name)


In [ ]:
# Ensure sample data exists
data_dirs = ["data/ohrc", "data/tmc", "data/iirc", "data/quickmap-lroc", "data/reference"]
for d in data_dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

# Generate synthetic dataset if empty
existing_files = list(Path("data").glob("*/*.png"))
if len(existing_files) < 10:
    print("Populating initial lunar test datasets...")
    gen = LunarSyntheticGenerator(size=(512, 512), seed=42)
    gen.populate_sample_datasets("data")


In [ ]:
# 1. Run Comprehensive Dataset Scan
scanner = LunarDatasetScanner(data_dirs)
df_report, summary = scanner.scan()

print(f"Total Scanned Images: {summary['total_images_scanned']}")
print(f"Valid Frames: {summary['valid_images_count']}")
print(f"Corrupted Images: {summary['corrupted_images_count']}")
print(f"Duplicate Files: {summary['duplicate_images_count']}")
print(f"Mean Resolution: {summary['mean_resolution']}")
print(f"Mean Radiometric Brightness: {summary['mean_dataset_brightness']}")
print(f"Mean RMS Contrast Ratio: {summary['mean_dataset_contrast']}")

df_report.head()


In [ ]:
# 2. Visualizations: Resolution, Brightness & Contrast Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Resolution distribution
if not df_report.empty:
    axes[0].hist(df_report['width'], bins=10, color='#2563EB', edgecolor='black', alpha=0.7)
    axes[0].set_title("Image Width Distribution (px)", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Width (pixels)")
    axes[0].set_ylabel("Count")

    # Brightness histogram
    axes[1].hist(df_report['mean_brightness'], bins=15, color='#F59E0B', edgecolor='black', alpha=0.7)
    axes[1].set_title("Mean Brightness (0-255 DN)", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Mean Gray Level")

    # Contrast histogram
    axes[2].hist(df_report['contrast_ratio'], bins=15, color='#10B981', edgecolor='black', alpha=0.7)
    axes[2].set_title("Contrast Ratio Distribution", fontsize=13, fontweight='bold')
    axes[2].set_xlabel("RMS Contrast / Mean")

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/01_dataset_distributions.png", dpi=300)
plt.show()


In [ ]:
# 3. Interactive Random Image Gallery
sample_images = df_report[~df_report['is_corrupted']].sample(min(4, len(df_report)))

fig_gallery, ax_g = plt.subplots(1, len(sample_images), figsize=(16, 4))
for idx, (_, row) in enumerate(sample_images.iterrows()):
    img = LunarImageLoader.load_image(row['file_path'])
    ax_g[idx].imshow(img, cmap='gray')
    ax_g[idx].set_title(f"{row['folder']}\n{row['width']}x{row['height']} | DN:{row['mean_brightness']}", fontsize=10)
    ax_g[idx].axis('off')

plt.tight_layout()
plt.savefig("outputs/visualizations/01_random_gallery.png", dpi=300)
plt.show()


In [ ]:
# 4. Export Reports: CSV, JSON, and HTML Audit
os.makedirs("outputs/reports", exist_ok=True)

csv_path = "outputs/reports/dataset_report.csv"
json_path = "outputs/reports/dataset_summary.json"
html_path = "outputs/reports/dataset_analysis.html"

df_report.to_csv(csv_path, index=False)
with open(json_path, "w") as f:
    json.dump(summary, f, indent=4)

df_report.to_html(html_path, classes="table table-striped")
print(f"Exported:\n- {csv_path}\n- {json_path}\n- {html_path}")
